# Lesson 6.3 · PCA in practice

Four typical uses, and three situations in which PCA misleads.

**What you will learn**

- PCA inside a `Pipeline` (for a KNN, for a regression).
- PCA to **visualize** and to **remove noise**.
- The limits: units of measure, outliers, nonlinear structure, informative low-variance components.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits, make_circles
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## 1. Visualizing

The first two components give a 2D map of the data: it is the fastest way to "look at" a dataset with many columns.

In [ ]:
P = PCA(2).fit_transform(X)
fig, ax = plt.subplots(figsize=(6, 4.5))
sc = ax.scatter(*P.T, c=y, cmap="tab10", s=8)
ax.set(xlabel="PC1", ylabel="PC2"); plt.colorbar(sc, label="digit"); plt.show()
print("variance explained by the first two components:", PCA(2).fit(X).explained_variance_ratio_.sum().round(3))

Only 29% of the variance sits in the first two components: some digits (the 0, for example) form a well separated group, others stay mixed. A 2D map is a **projection**, and the distances in the map are not the true ones. It must be read with caution (exercise D2).

## 2. Removing noise

If the signal sits in the first components and the noise is spread over all of them, reconstructing with few components **filters** the noise.

In [ ]:
rng = np.random.default_rng(0)
noisy = X + rng.normal(0, 4, X.shape)                                    # strong noise (the digits have pixels from 0 to 16)
for k in [64, 30, 15, 8]:
    p = PCA(k).fit(noisy)
    reconstructed = p.inverse_transform(p.transform(noisy))
    print(f"{k:2d} components | error with respect to the clean image: {((reconstructed - X) ** 2).mean():.2f}")

With all 64 components the error is that of the noise (16); with 30 components it drops to 9.1 and with 15 to 7.1, less than half: the reconstruction with few components **is closer to the clean original than the noisy image is**. With too few
(8) the error goes up again (8.3) because part of the signal is lost too. It is the same bias-variance tradeoff, applied to the reconstruction.

## 3. Inside a pipeline

PCA always goes inside the `Pipeline`, so it is learned on the training part of every fold alone. With a KNN on these **clean** digits it does not improve the accuracy (0.986 on the 64 pixels, 0.983 with 20 components): it compresses to a third of the dimensions for a cost of a few tenths of a point. The advantage on noise you measure in exercise B2.

In [ ]:
for name, m in {"KNN on 64 pixels": KNeighborsClassifier(5),
                "PCA(20) + KNN": make_pipeline(PCA(20), KNeighborsClassifier(5)),
                "PCA(10) + KNN": make_pipeline(PCA(10), KNeighborsClassifier(5))}.items():
    print(f"{name:16s} accuracy {cross_val_score(m, X, y, cv=cv).mean():.3f}")

## 4. Warning: PCA is linear and does not look at `y`

Two concentric circles are not separable with a line, and no rotation (which is what PCA does) makes them separable: PCA cannot help. And if the signal sits in a low-variance direction, PCA discards it.

In [ ]:
Xc, yc = make_circles(1000, noise=0.05, factor=0.4, random_state=0)
for k in [1, 2]:
    m = make_pipeline(PCA(k), LogisticRegression())
    print(f"concentric circles, PCA({k}) + logistic: accuracy {cross_val_score(m, Xc, yc, cv=cv).mean():.2f}")

With one or two components (in 2D, `PCA(2)` is just a rotation) the accuracy stays at 0.47 and 0.46, that is at chance: the problem is not linear and linear PCA does not change that (see F2 for the nonlinear alternatives, KernelPCA).

> **Common mistake.** Using PCA as "selection of the best features": it orders the directions by variance, not by usefulness toward a target.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>When is it worth applying PCA before a supervised model?</summary>

When there are many correlated or noisy columns (KNN, unregularized linear regression, models sensitive to collinearity), when the data are too large, or for a first visualization. With trees and boosting almost never: they are already robust to redundant features and the rotation can even make things worse (trees split along the original axes).

</details>

<details><summary>Is PCA interpretable?</summary>

Less than the original columns: every component is a combination of all of them. You interpret it by looking at the loadings (`components_`) and asking which variables move together, but a model on the components loses the direct reading "this column does this".

</details>